# Emotion & Sentiment Model Evaluation for Victim Distress Prioritisation

This notebook evaluates the NLP component of the **AI-Powered Dynamic Mental Health Monitoring and Distress Prediction System for Victims of Atrocities**.

> [!WARNING]
> **CRITICAL CLINICAL & ETHICAL DISCLAIMER**  
> **A text-emotion model cannot determine mental health condition; outputs are predictions, not diagnoses.**  
> This system is strictly an experimental support-prioritisation workflow aid for authorized human counsellors, NOT a diagnostic instrument.

In [ ]:
# Step 0: Install dependencies in Google Colab
!pip install -q transformers datasets scikit-learn matplotlib

## 1. Load the GoEmotions Dataset & Document Limitations

### Documented Dataset Limitations:
1. **Reddit Text Source:** Curated entirely from Reddit comment threads (2005–2019), representing internet colloquialisms, sarcasm, and meme idioms rather than real victim disclosures.
2. **Not Victim Data:** Does not originate from survivors of mass atrocities, hate violence, or human rights violations.
3. **Not Clinical:** Labelled by crowdsourced Mechanical Turk workers rather than trauma clinicians or psychiatric professionals.
4. **Label Noise & Ambiguity:** Significant overlap between neighboring labels (e.g. *annoyance* vs *anger*, *grief* vs *sadness*).

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load simplified config of GoEmotions
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
raw_labels = dataset["test"].features["labels"].feature.names
print(f"Test instances: {len(dataset['test'])}")
print(f"Raw GoEmotions labels ({len(raw_labels)}): {raw_labels}")

## 2. Emotion Grouping: 27 Raw Labels into 6 Target Classes

To support transparent distress triage, the 27 fine-grained emotion labels are grouped into 6 categories:
- **`fear/worry`**: fear, nervousness
- **`sadness`**: sadness, grief, remorse, disappointment, embarrassment
- **`anger`**: anger, annoyance, disapproval
- **`joy`**: joy, amusement, approval, excitement, gratitude, love, optimism, pride, relief
- **`neutral`**: neutral
- **`other`**: admiration, caring, confusion, curiosity, desire, realization, surprise, disgust

In [ ]:
GO_EMOTIONS_MAPPING = {
    "fear": "fear/worry", "nervousness": "fear/worry",
    "sadness": "sadness", "grief": "sadness", "remorse": "sadness",
    "disappointment": "sadness", "embarrassment": "sadness",
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    "joy": "joy", "amusement": "joy", "approval": "joy", "excitement": "joy",
    "gratitude": "joy", "love": "joy", "optimism": "joy", "pride": "joy", "relief": "joy",
    "neutral": "neutral",
    "admiration": "other", "caring": "other", "confusion": "other", "curiosity": "other",
    "desire": "other", "realization": "other", "surprise": "other", "disgust": "other"
}

def map_emotion(raw_label):
    return GO_EMOTIONS_MAPPING.get(raw_label.lower().strip(), "other")

## 3. Load Pretrained Models & Evaluate on GoEmotions Test Split

We evaluate:
1. **Emotion:** `SamLowe/roberta-base-go_emotions` (fine-tuned RoBERTa on GoEmotions).
2. **Sentiment:** `cardiffnlp/twitter-roberta-base-sentiment-latest` mapped to positive/neutral/negative.

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Load pipelines
emo_pipe = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions", device=-1)
sent_pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", device=-1)

# Prepare single-label test split
test_ds = dataset["test"]
eval_samples = []
for item in test_ds:
    if len(item["labels"]) == 1:
        raw = raw_labels[item["labels"][0]]
        eval_samples.append({"text": item["text"], "raw": raw, "grouped": map_emotion(raw)})
    if len(eval_samples) >= 250:
        break

texts = [s["text"] for s in eval_samples]
y_true = [s["grouped"] for s in eval_samples]

# Batch inference
preds = emo_pipe(texts, truncation=True, max_length=128)
y_pred = [map_emotion(p["label"]) for p in preds]

target_classes = ["fear/worry", "sadness", "anger", "joy", "neutral", "other"]
acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=target_classes, average="macro")
print(f"Emotion Evaluation Accuracy: {acc:.4f}")
print(f"Emotion Evaluation Macro F1: {macro_f1:.4f}")
print("\nPer-Class Classification Report:\n", classification_report(y_true, y_pred, labels=target_classes))

## 4. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=target_classes)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(len(target_classes)),
    yticks=np.arange(len(target_classes)),
    xticklabels=target_classes,
    yticklabels=target_classes,
    title="GoEmotions 6-Class Grouped Confusion Matrix",
    ylabel="True Label",
    xlabel="Predicted Label"
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
thresh = cm.max() / 2.
for i in range(len(target_classes)):
    for j in range(len(target_classes)):
        ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center", color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()

## 5. Evaluation on 15 Hand-Written Synthetic Check-In Sentences

We evaluate 15 synthetic statements reflecting realistic victim check-in disclosures across fear, threat, grief, anger, hope, and baseline functioning.

In [ ]:
synthetic_sentences = [
    ("I am worried about the upcoming court hearing, and I have not been sleeping properly.", "Anticipatory anxiety / Insomnia"),
    ("They drove past my house again last night shouting threats. I feel terrified.", "Active intimidation / Threat"),
    ("I feel completely numb and empty inside. Nothing seems to matter anymore.", "Depressive numbness / Helplessness"),
    ("I cannot stop crying today. The grief is overwhelming and heavy.", "Acute grief / Somatic bereavement"),
    ("I am so furious at the perpetrators walking free while we live in hiding.", "Trauma-related rage / Injustice"),
    ("I met with the legal aid advocate today and felt a glimmer of hope.", "Positive resilience / Advocacy connection"),
    ("We moved to a safehouse and for the first time in months I felt safe.", "Relief / Physical safety restored"),
    ("Every sudden noise outside sends my heart racing with panic.", "Hyperarousal / PTSD startle response"),
    ("I feel isolated from everyone in my community since the incident happened.", "Social stigma / Alienation"),
    ("The nightmares keep waking me up in cold sweats, but I am trying to cope.", "Trauma intrusion / Coping effort"),
    ("I need to talk to someone right now, I cannot manage these panic attacks alone.", "Acute crisis / Voluntary outreach"),
    ("Things are quiet today. Just doing normal chores at home.", "Routine baseline / Neutral activity"),
    ("I am deeply grateful for the counsellor checking in on me and my family.", "Social gratitude / Supportive bond"),
    ("I feel so bitter and resentful about everything that was taken from us.", "Resentment / Long-term grievance"),
    ("I managed to take a short walk outside today without feeling completely overwhelmed.", "Behavioral recovery milestone"),
]

results = []
for text, context in synthetic_sentences:
    e = emo_pipe(text)[0]
    s = sent_pipe(text)[0]
    raw_s = s["label"].lower()
    s_lbl = "negative" if ("neg" in raw_s or raw_s == "label_0") else ("positive" if ("pos" in raw_s or raw_s == "label_2") else "neutral")
    results.append({
        "Sentence": text,
        "Context": context,
        "Sentiment": f"{s_lbl} ({s['score']:.2f})",
        "Emotion": f"{map_emotion(e['label'])} ({e['score']:.2f})",
        "Raw Label": e["label"]
    })

df_results = pd.DataFrame(results)
df_results

## 6. Limitations and Ethical Use

1. **Non-Diagnostic:** Text classification cannot diagnose clinical depression, PTSD, or suicidality.
2. **Triage Prior Only:** Model inferences serve solely as numerical priors inside the pure-function distress scoring formula to support human counsellors.
3. **Human-in-the-Loop Oversight:** Every alert and case prioritisation score is reviewed and actionable only by an authorized human case worker.